# 5 — BankNifty Enhanced Charts V3 (Split Figures + ROC Average Line)

**Run frequency:** During market hours alongside Notebook 2

**What this does:**
1. Logs in to Zerodha KiteConnect (needed for live ATM strike)
2. Starts a SparkSession
3. Reads instruments + expiries from Parquet
4. Builds the options DataFrame for the configured ATM strike
5. Renders **enhanced V3** Plotly charts:

   | Figure | Layout | Content |
   |--------|--------|---------|
   | 1 | 1×2 | **Straddle Price & Signals** \| **ROC AVG + ROC Average Line** |
   | 2 | 1×2 | **CE vs PE Close** \| **CE/PE ROC & Ratio** *(straddle only)* |
   | 3 | 1×2 | Combined OI \| CE/PE OI + PUT−CALL OI secondary Y |
   | 4 | standalone | PCR |
   | 5 | 2×1 shared-x | Straddle Price/VWAP/OIWAP + PUT−CALL OI indicator pane |
   | 6 | 2×1 shared-x | OHLC Candlestick + Volume |

   All charts use `DD-Mon HH:MM` time labels with vertical day-boundary lines.

6. **Auto-refreshes** every minute; stops automatically at `END_HOUR:END_MINUTE`

---
### ⚙️ Configuration

| Parameter | Default | Description |
|-----------|---------|-------------|
| `STRIKE_LEVEL_NAME` | `'ATM'` | Which strike to chart |
| `CE_OR_PE` | `'E'` | `'CE'`, `'PE'`, or `'E'` for straddle |
| `NUM_DAYS` | `2` | Days of data to display |
| `IS_LATEST_DAY` | `True` | Show only today's data |
| `CUSTOM_STRIKE` | `0` | Override ATM (0 = live LTP) |
| `NUM_STRIKES` | `11` | Strikes each side of ATM to load |
| `LOOP_INTERVAL_MIN` | `1` | Chart refresh frequency (minutes) |
| `END_HOUR` | `15` | Stop loop at this hour (24h, set `None` to run indefinitely) |
| `END_MINUTE` | `30` | Stop loop at this minute |

In [1]:
# ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
# os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [2]:
# ── CONFIG — edit these values ─────────────────────────────────────────────
STRIKE_LEVEL_NAME  = 'ATM'   # 'ATM', 'ATM+1', 'ATM-2', etc.
CE_OR_PE           = 'E'     # 'CE', 'PE', or 'E' for straddle
NUM_DAYS           = 2       # Days of OHLC data to show
IS_LATEST_DAY      = True    # True = show today only; False = show NUM_DAYS
NUM_DAYS_BACK      = 0       # 0 = latest, 1 = go back 1 day
CUSTOM_STRIKE      = 56500   # 0 = use live LTP; override e.g. 56500
NUM_STRIKES        = 11      # Strikes each side of ATM to load
LOOP_INTERVAL_MIN  = 1       # Chart refresh frequency (minutes)
INTERVAL           = '3minute'

# Time-based loop exit — set to None to run indefinitely
END_HOUR           = 15      # 24-hour format (e.g. 15 = 3 PM)
END_MINUTE         = 30      # Minute (e.g. 30 = :30)

In [3]:
# ── Step 1: Login ──────────────────────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print('✅ Login successful')

2026-03-12 00:01:42,640 [INFO] utils.kite_helpers — Logging in as user: ZB4988
2026-03-12 00:01:42,830 [INFO] utils.kite_helpers — Generated TOTP pin for 2FA
2026-03-12 00:01:43,006 [INFO] utils.kite_helpers — Login successful — request_token: jXSEJN5G2WgHHZaTAL09MKOgfzybFyei
2026-03-12 00:01:43,222 [INFO] utils.kite_helpers — KiteConnect session established for user: ZB4988


✅ Login successful


In [4]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name='5_Enhanced_Charts_BNF')
print('✅ Spark session ready')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/12 00:01:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark session ready


In [5]:
# ── Step 3: Load instruments and expiry data ───────────────────────────────
from utils.strike_utils import read_instruments, get_expiry_dates, get_Options_DF, get_ATM_Strike
from utils.strike_utils import BANKNIFTY_INDEX_TOKEN

bnf_options, bnf_futures, nifty_options, nifty_futures, expiries_df = read_instruments(spark)
bnf_expiries = get_expiry_dates(expiries_df, 'BANKNIFTY')

print(f"BankNifty current week expiry: {bnf_expiries['current_week']}")
print(f"BankNifty next week expiry:    {bnf_expiries['next_week']}")

BankNifty current week expiry: 2026-03-30
BankNifty next week expiry:    2026-04-28


In [6]:
# ── Step 4: Build options DataFrame for ATM region ────────────────────────
if CUSTOM_STRIKE > 0:
    bnf_atm = CUSTOM_STRIKE
else:
    bnf_atm = get_ATM_Strike(kite, BANKNIFTY_INDEX_TOKEN, rounding_number=100)

print(f'BankNifty ATM Strike = {bnf_atm}')

Banknifty_Options_DF = get_Options_DF(
    spark                = spark,
    options_df_from_file = bnf_options,
    atm_strike           = bnf_atm,
    current_expiry       = bnf_expiries['current_week'],
    strike_range         = 100,
    num_strikes          = NUM_STRIKES,
)
print(f"Options loaded: {Banknifty_Options_DF.count()} rows")
Banknifty_Options_DF.show(5)

BankNifty ATM Strike = 56500
Options loaded: 46 rows


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py:154: DeprecationWarning: This process (pid=89708) is multi-threaded, use of fork() may lead to deadlocks in the child.


+---------+----------------+----------+-----------------+-------+-----------------+--------+---------------+-------+
|     name|instrument_token|    expiry|Strike_level_Name| strike|Strike_level_type|lot_size|instrument_type|segment|
+---------+----------------+----------+-----------------+-------+-----------------+--------+---------------+-------+
|BANKNIFTY|        13430786|2026-03-30|           ATM-11|55400.0|              ITM|      30|             CE|NFO-OPT|
|BANKNIFTY|        13431042|2026-03-30|           ATM-11|55400.0|              ITM|      30|             PE|NFO-OPT|
|BANKNIFTY|        14988290|2026-03-30|           ATM-10|55500.0|              ITM|      30|             CE|NFO-OPT|
|BANKNIFTY|        15002114|2026-03-30|           ATM-10|55500.0|              ITM|      30|             PE|NFO-OPT|
|BANKNIFTY|        13431298|2026-03-30|            ATM-9|55600.0|              ITM|      30|             CE|NFO-OPT|
+---------+----------------+----------+-----------------+-------

In [ ]:
# ── Step 5: Start auto-refreshing ENHANCED V3 charts ──────────────────────
# This cell blocks until END_HOUR:END_MINUTE or Kernel → Interrupt.
#
# Charts rendered each refresh:
#   Fig 1 — Price & Signals  (1×2: Straddle+BB+SL | ROC AVG + ROC Average Line)
#   Fig 2 — CE / PE panels   (1×2: CE/PE Close | CE/PE ROC+Ratio)  [straddle only]
#   Fig 3 — Open Interest    (1×2: Combined OI | CE/PE OI + PUT−CALL OI secondary Y)
#   Fig 4 — PCR              (standalone)
#   Fig 5 — Straddle Price / VWAP / OIWAP  +  PUT−CALL OI indicator pane
#   Fig 6 — OHLC Candlestick + Volume
from utils.enhanced_charts_v3 import run_enhanced_chart_loop_v3

run_enhanced_chart_loop_v3(
    spark                 = spark,
    options_df            = Banknifty_Options_DF,
    expiry                = bnf_expiries['current_week'],
    strike_level_name     = STRIKE_LEVEL_NAME,
    ce_or_pe              = CE_OR_PE,
    interval              = INTERVAL,
    num_days              = NUM_DAYS,
    is_latest_day         = IS_LATEST_DAY,
    loop_interval_minutes = LOOP_INTERVAL_MIN,
    end_hour              = END_HOUR,
    end_minute            = END_MINUTE,
)


📸 Charts saved to ChartsSnapshots/2026-03-12/ at 00:02


2026-03-12 00:03:34,169 [ERROR] root — KeyboardInterrupt while sending command.]
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socket.py", line 708, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
2026-03-12 00:03:34,181 [INFO] py4j.clientserver — Closing down clientserver connection
2026-03-12 00:03:34,198 [INFO] utils.enhanced_charts_v3 — Enhanced chart v2 loop stopped.
2026-03-12 00:03:34,199 [INFO] py4j

26/03/12 00:03:34 ERROR Executor: Exception in task 0.0 in stage 60.0 (TID 37)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/worker.py", line 1094, in main
    split_index = read_int(infile)
                  ^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/serializers.py", line 594, in read_int
    length = stream.read(4)
             ^^^^^^^^^^^^^^
KeyboardInterrupt

	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator.handlePythonException(PythonRunner.scala:572)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:94)
	at org.apache.spark.sql.execution.python.BasePythonUDFRunner$$anon$1.read(PythonUDFRunner.scala:75)
	at org.apache.spark.api.python.BasePythonRunner$ReaderItera